# RandLANet Classification Debug with Pytorch Geometric

In [1]:
%load_ext autoreload
%autoreload 2

## Imports

In [2]:
import time
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import open3d.ml as _ml3d
import ml3d.torch as ml3d
from ml3d.torch.dataloaders import get_sampler, TorchDataloader
from ml3d.datasets import InferenceDummySplit

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Original Implementation

In [40]:
cfg_file = "/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/configs/randlanet_semantickitti.yml"
cfg = _ml3d.utils.Config.load_from_file(cfg_file)
cfg.model.num_points = 10_000

In [41]:
model = ml3d.models.RandLANet(**cfg.model)
cfg.dataset['dataset_path'] = '/media/arthur/HDD/Datasets/Point Clouds/SemanticKitti/raw/'
dataset = _ml3d.datasets.SemanticKITTI(cfg.dataset.pop('dataset_path', None), **cfg.dataset)
pipeline = ml3d.pipelines.SemanticSegmentation(model, dataset=dataset, device="gpu", **cfg.pipeline)

test 0/1: 100%|██████████| 4257/4257 [03:00<00:00, 23.62it/s]


In [42]:
# download the weights.
ckpt_folder = "./logs/"
os.makedirs(ckpt_folder, exist_ok=True)
ckpt_path = ckpt_folder + "randlanet_semantickitti_202201071330utc.pth"
randlanet_url = "https://storage.googleapis.com/open3d-releases/model-zoo/randlanet_semantickitti_202201071330utc.pth"
if not os.path.exists(ckpt_path):
    cmd = "wget {} -O {}".format(randlanet_url, ckpt_path)
    os.system(cmd)

In [43]:
# load the parameters.
pipeline.load_ckpt(ckpt_path=ckpt_path)

In [44]:
test_split = dataset.get_split("test")
data = test_split.get_data(0)
data

{'point': array([[ 1.35929756e+01,  7.97351729e-03,  6.69000268e-01],
        [ 1.35728445e+01,  5.08311652e-02,  6.68001473e-01],
        [ 1.35917816e+01,  7.17616528e-02,  6.69002056e-01],
        ...,
        [ 7.93910265e+00, -2.63877869e+00, -3.77945089e+00],
        [ 7.96492624e+00, -2.61984015e+00, -3.78844428e+00],
        [ 7.98275280e+00, -2.59890866e+00, -3.79343700e+00]], dtype=float32),
 'feat': None,
 'label': array([0, 0, 0, ..., 0, 0, 0], dtype=int32)}

In [54]:
# Generate dummy data
torch.random.manual_seed(0)
data = {
    "point": torch.rand((10_000, 3)),
    "feat": None,
    "label": torch.zeros(10_000, dtype=torch.long),
}

In [55]:
# run inference on a single example.
# returns dict with 'predict_labels' and 'predict_scores'.
result = pipeline.run_inference(data)

test 0/1: 100%|██████████| 4257/4257 [24:15<00:00,  2.92it/s]
/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/pipelines/semantic_segmentation.py:173: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(data['label']), model.cfg.num_classes,
/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/modules/metrics/semseg_metric.py:54: RuntimeWarning: Mean of empty slice
  accs.append(np.nanmean(accs))
/home/arthur/Documents/Code/Github/Open3D-ML/ml3d/torch/modules/metrics/semseg_metric.py:87: RuntimeWarning: Mean of empty slice
  ious.append(np.nanmean(ious))


In [69]:
batcher = pipeline.get_batcher("cpu")
infer_dataset = InferenceDummySplit(data)
pipeline.dataset_split = infer_dataset

infer_sampler = infer_dataset.sampler
infer_split = TorchDataloader(dataset=infer_dataset,
                                preprocess=model.preprocess,
                                transform=model.transform,
                                sampler=infer_sampler,
                                use_cache=False)

infer_loader = DataLoader(infer_split,
                            batch_size=1,
                            sampler=get_sampler(infer_sampler),
                            collate_fn=batcher.collate_fn)

model.trans_point_sampler = infer_sampler.get_point_sampler()

In [73]:
with torch.no_grad():
    for _, inputs in enumerate(infer_loader):
        results = model(inputs["data"])
        break

DECODER [i = 0] feat.shape = torch.Size([1, 512, 39, 1])
DECODER [i = 0] interpolation_indices_list[-i - 1].shape = torch.Size([1, 156, 1])
DECODER [i = 0] encoder_feat_list[-i - 2].shape = torch.Size([1, 256, 156, 1])
DECODER [i = 0] feat_interpolation_i.shape = torch.Size([1, 512, 156, 1])
DECODER [i = 1] feat.shape = torch.Size([1, 256, 156, 1])
DECODER [i = 1] interpolation_indices_list[-i - 1].shape = torch.Size([1, 625, 1])
DECODER [i = 1] encoder_feat_list[-i - 2].shape = torch.Size([1, 128, 625, 1])
DECODER [i = 1] feat_interpolation_i.shape = torch.Size([1, 256, 625, 1])
DECODER [i = 2] feat.shape = torch.Size([1, 128, 625, 1])
DECODER [i = 2] interpolation_indices_list[-i - 1].shape = torch.Size([1, 2500, 1])
DECODER [i = 2] encoder_feat_list[-i - 2].shape = torch.Size([1, 32, 2500, 1])
DECODER [i = 2] feat_interpolation_i.shape = torch.Size([1, 128, 2500, 1])
DECODER [i = 3] feat.shape = torch.Size([1, 32, 2500, 1])
DECODER [i = 3] interpolation_indices_list[-i - 1].shape = 

In [76]:
model.decoder

ModuleList(
  (0): SharedMLP(
    (conv): ConvTranspose2d(768, 256, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(256, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (1): SharedMLP(
    (conv): ConvTranspose2d(384, 128, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(128, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (2): SharedMLP(
    (conv): ConvTranspose2d(160, 32, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(32, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (3): SharedMLP(
    (conv): ConvTranspose2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(32, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
)

In [77]:
model.fc1

Sequential(
  (0): SharedMLP(
    (conv): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(64, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (1): SharedMLP(
    (conv): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(32, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (2): Dropout(p=0.5, inplace=False)
  (3): SharedMLP(
    (conv): Conv2d(32, 19, kernel_size=(1, 1), stride=(1, 1))
  )
)

In [59]:
inputs["data"]["coords"][0].shape

torch.Size([1, 10000, 3])

In [71]:
for i, tensor in enumerate(inputs["data"]["coords"]):
    print(f"{i}: {tensor.shape}")

0: torch.Size([1, 10000, 3])
1: torch.Size([1, 2500, 3])
2: torch.Size([1, 625, 3])
3: torch.Size([1, 156, 3])


In [72]:
for i, tensor in enumerate(inputs["data"]["interp_idx"]):
    print(f"{i}: {tensor.shape}")

0: torch.Size([1, 10000, 1])
1: torch.Size([1, 2500, 1])
2: torch.Size([1, 625, 1])
3: torch.Size([1, 156, 1])


In [4]:
import torch

target = torch.tensor([0, 1, 1, 2, 2, 2])
torch.bincount(target)

# Same for segmentation
target = torch.tensor(
    [
        [0, 1, 1, 2, 2, 2],
        [0, 1, 1, 2, 2, 2],
    ]
)

torch.bincount(target.flatten())

tensor([2, 4, 6])

In [14]:
model.decoder

ModuleList(
  (0): SharedMLP(
    (conv): ConvTranspose2d(768, 256, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(256, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (1): SharedMLP(
    (conv): ConvTranspose2d(384, 128, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(128, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (2): SharedMLP(
    (conv): ConvTranspose2d(160, 32, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(32, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
  (3): SharedMLP(
    (conv): ConvTranspose2d(64, 32, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(32, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
)

## Open3D Blocks

In [15]:
class SharedMLP(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=1,
        stride=1,
        transpose=False,
        bn=True,
        activation_fn=None,
    ):
        super(SharedMLP, self).__init__()

        if transpose:
            self.conv = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=(kernel_size - 1) // 2,
            )
        else:
            self.conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=(kernel_size - 1) // 2,
            )

        self.batch_norm = (
            nn.BatchNorm2d(out_channels, eps=1e-6, momentum=0.01) if bn else None
        )
        self.activation_fn = activation_fn

    def forward(self, input):
        """Forward pass of the Module.

        Args:
            input: torch.Tensor of shape (B, dim_in, N, K)

        Returns:
            torch.Tensor, shape (B, dim_out, N, K)

        """
        x = self.conv(input)
        if self.batch_norm:
            x = self.batch_norm(x)
        if self.activation_fn:
            x = self.activation_fn(x)
        return x


class LocalSpatialEncoding(nn.Module):
    def __init__(self, dim_in, dim_out, num_neighbors, encode_pos=False):
        super(LocalSpatialEncoding, self).__init__()

        self.num_neighbors = num_neighbors
        self.mlp = SharedMLP(dim_in, dim_out, activation_fn=nn.LeakyReLU(0.2))
        self.encode_pos = encode_pos

    def gather_neighbor(self, coords, neighbor_indices):
        """Gather features based on neighbor indices.

        Args:
            coords: torch.Tensor of shape (B, N, d)
            neighbor_indices: torch.Tensor of shape (B, N, K)

        Returns:
            gathered neighbors of shape (B, dim, N, K)

        """
        B, N, K = neighbor_indices.size()
        dim = coords.shape[2]

        extended_indices = neighbor_indices.unsqueeze(1).expand(B, dim, N, K)
        extended_coords = coords.transpose(-2, -1).unsqueeze(-1).expand(B, dim, N, K)
        neighbor_coords = torch.gather(
            extended_coords, 2, extended_indices
        )  # (B, dim, N, K)

        return neighbor_coords

    def forward(self, coords, features, neighbor_indices, relative_features=None):
        """Forward pass of the Module.

        Args:
            coords: coordinates of the pointcloud
                torch.Tensor of shape (B, N, 3)
            features: features of the pointcloud.
                torch.Tensor of shape (B, d, N, 1)
            neighbor_indices: indices of k neighbours.
                torch.Tensor of shape (B, N, K)
            relative_features: relative neighbor features calculated
              on first pass. Required for second pass.

        Returns:
            torch.Tensor of shape (B, 2*d, N, K)

        """
        # finding neighboring points
        B, N, K = neighbor_indices.size()

        if self.encode_pos:
            neighbor_coords = self.gather_neighbor(coords, neighbor_indices)

            extended_coords = coords.transpose(-2, -1).unsqueeze(-1).expand(B, 3, N, K)

            relative_pos = extended_coords - neighbor_coords
            relative_dist = torch.sqrt(
                torch.sum(torch.square(relative_pos), dim=1, keepdim=True)
            )

            relative_features = torch.cat(
                [relative_dist, relative_pos, extended_coords, neighbor_coords], dim=1
            )

        else:
            if relative_features is None:
                raise ValueError(
                    "LocalSpatialEncoding: Require relative_features for second pass."
                )

        relative_features = self.mlp(relative_features)

        neighbor_features = self.gather_neighbor(
            features.transpose(1, 2).squeeze(3), neighbor_indices
        )

        return (
            torch.cat([neighbor_features, relative_features], dim=1),
            relative_features,
        )


class AttentivePooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(AttentivePooling, self).__init__()

        self.score_fn = nn.Sequential(
            nn.Linear(in_channels, in_channels), nn.Softmax(dim=-2)
        )
        self.mlp = SharedMLP(in_channels, out_channels, activation_fn=nn.LeakyReLU(0.2))

    def forward(self, x):
        """Forward pass of the Module.

        Args:
            x: torch.Tensor of shape (B, dim_in, N, K).

        Returns:
            torch.Tensor of shape (B, d_out, N, 1).

        """
        # computing attention scores
        scores = self.score_fn(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

        # sum over the neighbors
        features = torch.sum(scores * x, dim=-1, keepdim=True)  # shape (B, d_in, N, 1)

        return self.mlp(features)


class LocalFeatureAggregation(nn.Module):
    def __init__(self, d_in, d_out, num_neighbors):
        super(LocalFeatureAggregation, self).__init__()

        self.num_neighbors = num_neighbors

        self.mlp1 = SharedMLP(d_in, d_out // 2, activation_fn=nn.LeakyReLU(0.2))
        self.lse1 = LocalSpatialEncoding(10, d_out // 2, num_neighbors, encode_pos=True)
        self.pool1 = AttentivePooling(d_out, d_out // 2)

        self.lse2 = LocalSpatialEncoding(d_out // 2, d_out // 2, num_neighbors)
        self.pool2 = AttentivePooling(d_out, d_out)
        self.mlp2 = SharedMLP(d_out, 2 * d_out)

        self.shortcut = SharedMLP(d_in, 2 * d_out)
        self.lrelu = nn.LeakyReLU()

    def forward(self, coords, feat, neighbor_indices):
        """Forward pass of the Module.

        Args:
            coords: coordinates of the pointcloud
                torch.Tensor of shape (B, N, 3).
            feat: features of the pointcloud.
                torch.Tensor of shape (B, d, N, 1)
            neighbor_indices: Indices of neighbors.

        Returns:
            torch.Tensor of shape (B, 2*d_out, N, 1).

        """
        x = self.mlp1(feat)

        x, neighbor_features = self.lse1(coords, x, neighbor_indices)
        x = self.pool1(x)

        x, _ = self.lse2(
            coords, x, neighbor_indices, relative_features=neighbor_features
        )
        x = self.pool2(x)

        return self.lrelu(self.mlp2(x) + self.shortcut(feat))

In [16]:
# ['mlp1.conv.weight',
#  'mlp1.conv.bias',
#  'mlp1.batch_norm.weight',
#  'mlp1.batch_norm.bias',
#  'mlp1.batch_norm.running_mean',
#  'mlp1.batch_norm.running_var',
#  'mlp1.batch_norm.num_batches_tracked',
#  'lse1.mlp.conv.weight',
#  'lse1.mlp.conv.bias',
#  'lse1.mlp.batch_norm.weight',
#  'lse1.mlp.batch_norm.bias',
#  'lse1.mlp.batch_norm.running_mean',
#  'lse1.mlp.batch_norm.running_var',
#  'lse1.mlp.batch_norm.num_batches_tracked',
#  'pool1.score_fn.0.weight',
#  'pool1.score_fn.0.bias',
#  'pool1.mlp.conv.weight',
#  'pool1.mlp.conv.bias',
#  'pool1.mlp.batch_norm.weight',
#  'pool1.mlp.batch_norm.bias',
#  'pool1.mlp.batch_norm.running_mean',
#  'pool1.mlp.batch_norm.running_var',
#  'pool1.mlp.batch_norm.num_batches_tracked',
#  'lse2.mlp.conv.weight',
#  'lse2.mlp.conv.bias',
#  'lse2.mlp.batch_norm.weight',
#  'lse2.mlp.batch_norm.bias',
#  'lse2.mlp.batch_norm.running_mean',
#  'lse2.mlp.batch_norm.running_var',
#  'lse2.mlp.batch_norm.num_batches_tracked',
#  'pool2.score_fn.0.weight',
#  'pool2.score_fn.0.bias',
#  'pool2.mlp.conv.weight',
#  'pool2.mlp.conv.bias',
#  'pool2.mlp.batch_norm.weight',
#  'pool2.mlp.batch_norm.bias',
#  'pool2.mlp.batch_norm.running_mean',
#  'pool2.mlp.batch_norm.running_var',
#  'pool2.mlp.batch_norm.num_batches_tracked',
#  'mlp2.conv.weight',
#  'mlp2.conv.bias',
#  'mlp2.batch_norm.weight',
#  'mlp2.batch_norm.bias',
#  'mlp2.batch_norm.running_mean',
#  'mlp2.batch_norm.running_var',
#  'mlp2.batch_norm.num_batches_tracked',
#  'shortcut.conv.weight',
#  'shortcut.conv.bias',
#  'shortcut.batch_norm.weight',
#  'shortcut.batch_norm.bias',
#  'shortcut.batch_norm.running_mean',
#  'shortcut.batch_norm.running_var',
#  'shortcut.batch_norm.num_batches_tracked']

test 0/1: 100%|██████████| 79845/79845 [00:19<00:00, 82481.56it/s]

### Test LocalFeatureAggregation